In [6]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [10]:
from langchain_community.document_loaders import PyPDFLoader
PDF_Path = "telecom_guide.pdf"
loader = PyPDFLoader(PDF_Path)
pages = loader.load()

print(f"Loaded {len(pages)} pages")
print("\n --- Last Page ---")
print(pages[-1].page_content[:500])

Loaded 9 pages

 --- Last Page ---
Telecom Technical Reference Guide  - Internal Use Only
8. Network Security and Fraud Prevention
Telecom networks are high-value targets for fraud. Agents play a critical role in the first line of defence.
SS7 and Diameter Vulnerabilities: The inter-operator signalling protocols (SS7 for 2G/3G and Diameter for 4G)
were designed for trusted networks and have well-known vulnerabilities. Attackers with access to SS7 can
intercept SMS messages (undermining SMS-based 2FA), track device location, and r


In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_documents(pages)
print(f"Created {len(chunks)} chunks")

Created 45 chunks


In [14]:
print(chunks[1].page_content)

Telecom Technical Reference Guide  - Internal Use Only
1. Introduction to Mobile Networks
Mobile networks have evolved through several generations, each offering significant improvements in speed,
capacity, and capability.
2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to
around 50 kbps, sufficient only for text messaging and simple email.


In [16]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma 

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5162.37it/s]


In [17]:
print(f"vector store ready. {vector_store._collection.count()} vectors")

vector store ready. 45 vectors


In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
test_query = "What is VoWiFi?"
results = retriever.invoke(test_query)
print(f"Query: {test_query}")
print(f"Retrieved {len(results)} documents")
for i, doc in enumerate(results, 1):
    print(f"\nDocument {i}:")
    print(f"Content: {doc.page_content[:100]}...")

Query: What is VoWiFi?
Retrieved 3 documents

Document 1:
Content: the device may not support VoLTE or the profile has not been pushed to the SIM. Agents can push the ...

Document 2:
Content: Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services...

Document 3:
Content: in smartwatches and IoT devices.
SIM Security: The SIM stores Ki, a 128-bit authentication key that ...


In [25]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

SYSTEM_PROMPT = """
You are a helpful telecom assistant. Use the provided context to answer questions.
If the answer is not in the context, say you don't know.
Be concise and accurate.

Context Format:
{context}

Question: {question}
"""

def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}")
])

llm = ChatGroq(model_name="llama-3.1-8b-instant",temperature=0)
chain = ({'context': retriever | format_docs, 'question': RunnablePassthrough()} | prompt | llm | StrOutputParser())

print("RAG Chain created successfully!")

RAG Chain created successfully!


In [26]:
question = "How does Volte work? and what are its benefits?"

print(f"Question: {question}")
print(f"Answer: {chain.invoke(question)}")

Question: How does Volte work? and what are its benefits?


Answer: VoLTE works by transmitting voice calls as data packets over the LTE network using the IMS (IP Multimedia Subsystem) core. 

Its benefits include:

1. HD voice quality (wideband audio at 16 kHz)
2. Faster call setup times (under 2 seconds)
3. Ability to use data and voice simultaneously without degradation.
